# 1. LIBRARIES 

In [1]:
from pathlib import Path
import json

import torch
import pandas as pd

import cv2
cv2.setNumThreads(0)

from ultralytics import YOLO

In [2]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: 0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


# 2. LOAD PRE-TRAINED MODEL YOLO11N-SEG

In [3]:
MODEL_NAME = "yolo11n-seg.pt"
model = YOLO(MODEL_NAME)

model.info()


YOLO11n-seg summary: 203 layers, 2,876,848 parameters, 0 gradients, 10.0 GFLOPs


(203, 2876848, 0, 9.9593344)

# 3. INSPECT MODEL LAYER 

In [4]:
for i, layer in enumerate(model.model.model):
    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__}"
    )

00 | Conv
01 | Conv
02 | C3k2
03 | Conv
04 | C3k2
05 | Conv
06 | C3k2
07 | Conv
08 | C3k2
09 | SPPF
10 | C2PSA
11 | Upsample
12 | Concat
13 | C3k2
14 | Upsample
15 | Concat
16 | C3k2
17 | Conv
18 | Concat
19 | C3k2
20 | Conv
21 | Concat
22 | C3k2
23 | Segment


In [5]:
print("=== MODEL PARAMETER DIAGNOSTIC ===\n")

total = 0
trainable = 0
frozen = 0

for name, param in model.model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()
    else:
        frozen += param.numel()

    print(
        f"{name:70s} | "
        f"requires_grad={param.requires_grad} | "
        f"shape={tuple(param.shape)}"
    )

print("\n=== SUMMARY ===")
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters:    {frozen:,}")

=== MODEL PARAMETER DIAGNOSTIC ===

model.0.conv.weight                                                    | requires_grad=False | shape=(16, 3, 3, 3)
model.0.bn.weight                                                      | requires_grad=False | shape=(16,)
model.0.bn.bias                                                        | requires_grad=False | shape=(16,)
model.1.conv.weight                                                    | requires_grad=False | shape=(32, 16, 3, 3)
model.1.bn.weight                                                      | requires_grad=False | shape=(32,)
model.1.bn.bias                                                        | requires_grad=False | shape=(32,)
model.2.cv1.conv.weight                                                | requires_grad=False | shape=(32, 32, 1, 1)
model.2.cv1.bn.weight                                                  | requires_grad=False | shape=(32,)
model.2.cv1.bn.bias                                                    | requires_

In [6]:

for param in model.model.parameters():
    param.requires_grad = True

print("All parameters restored to trainable.")

All parameters restored to trainable.


In [7]:
trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model.model.parameters()
    if not p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

Total parameters:     2,876,848
Trainable parameters: 2,876,848
Frozen parameters:    0


# 4. FREEZE BACKBONE LAYER 

In [8]:
BACKBONE_END = 10
FREEZE_LAYERS = BACKBONE_END + 1

for i, layer in enumerate(model.model.model):
    if i <= BACKBONE_END:
        for param in layer.parameters():
            param.requires_grad = False

print("Backbone layers 00–10 frozen.")

Backbone layers 00–10 frozen.


In [9]:
for i, layer in enumerate(model.model.model):
    params = list(layer.parameters())

    if len(params) == 0:
        status = "NO PARAMETERS"
    else:
        trainable = any(p.requires_grad for p in params)
        status = "TRAINABLE" if trainable else "FROZEN"

    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__:12s} | "
        f"{status}"
    )

00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMETERS
12 | Concat       | NO PARAMETERS
13 | C3k2         | TRAINABLE
14 | Upsample     | NO PARAMETERS
15 | Concat       | NO PARAMETERS
16 | C3k2         | TRAINABLE
17 | Conv         | TRAINABLE
18 | Concat       | NO PARAMETERS
19 | C3k2         | TRAINABLE
20 | Conv         | TRAINABLE
21 | Concat       | NO PARAMETERS
22 | C3k2         | TRAINABLE
23 | Segment      | TRAINABLE


# 5. STAGE 1 FINE TUNING CONFIGURATION 

In [10]:
PROJECT_ROOT = Path("..").resolve()
DATASET_VERSION = "v2"
STAGE = "stage1"

In [11]:
DATASET_DIR = PROJECT_ROOT / "datasets" / DATASET_VERSION
DATA_YAML = DATASET_DIR / "data.yaml"

In [12]:
RUNS_DIR = PROJECT_ROOT / "runs"

EXPERIMENT_DIR = (
    RUNS_DIR
    / DATASET_VERSION
    / "segmentation"
    / "stage1"
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [13]:
STAGE1_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "device": DEVICE,
    "freeze": FREEZE_LAYERS,
    "workers": 2,          
    "save_period": 5,       

    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,

    "warmup_epochs": 3,
    "patience": 15,

    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 5.0, "translate": 0.1, "scale": 0.5, "fliplr": 0.5,
    "copy_paste": 0.4,
    "copy_paste_mode": "mixup",
    "erasing": 0.1,
    "mask_ratio": 2,

    "amp": True,
    "seed": 42,
}

In [14]:


print("Model:", "YOLO11n-Seg")
print("Dataset:", DATA_YAML)
print("Device:", DEVICE)

print("\nParameters:")

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total:      {total_params:,}")
print(f"Trainable:  {trainable_params:,}")
print(f"Frozen:     {frozen_params:,}")

print("\nExpected:")
print("Backbone 00–10 → FROZEN")
print("Neck 11–22     → TRAINABLE")
print("Head 23        → TRAINABLE")

Model: YOLO11n-Seg
Dataset: D:\PREP_INTERN\nutrivision_pro\datasets\v2\data.yaml
Device: 0

Parameters:
Total:      2,876,848
Trainable:  1,511,376
Frozen:     1,365,472

Expected:
Backbone 00–10 → FROZEN
Neck 11–22     → TRAINABLE
Head 23        → TRAINABLE


# 6. FINE TUNING 

In [15]:
LAST_CKPT = EXPERIMENT_DIR / "weights" / "last.pt"
stage1_results = model.train(
    **STAGE1_CONFIG,         
    resume=LAST_CKPT.exists(), 
    project=str(EXPERIMENT_DIR.parent),
    name=EXPERIMENT_DIR.name,
    exist_ok=True,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)  
print(f"Run saved in : {RUN_DIR}")
print(f"Best checkpoint : {RUN_DIR / 'weights' / 'best.pt'}")

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=mixup, cos_lr=False, cutmix=0.0, data=D:\PREP_INTERN\nutrivision_pro\datasets\v2\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=11, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=2, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

In [16]:
print("=== Verification after freeze the layer ===")
trained = YOLO(RUN_DIR / "weights" / "last.pt")  
for i, layer in enumerate(trained.model.model):
    params = list(layer.parameters())
    status = "NO PARAMS" if not params else ("TRAINABLE" if any(p.requires_grad for p in params) else "FROZEN")
    print(f"{i:02d} | {layer.__class__.__name__:12s} | {status}")

=== Verification after freeze the layer ===
00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMS
12 | Concat       | NO PARAMS
13 | C3k2         | FROZEN
14 | Upsample     | NO PARAMS
15 | Concat       | NO PARAMS
16 | C3k2         | FROZEN
17 | Conv         | FROZEN
18 | Concat       | NO PARAMS
19 | C3k2         | FROZEN
20 | Conv         | FROZEN
21 | Concat       | NO PARAMS
22 | C3k2         | FROZEN
23 | Segment      | FROZEN


In [17]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.14.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


# 7. VALIDATION FINE TUNING STAGE 1 

## 7.1. Load best.pt

In [18]:
from ultralytics import YOLO

# Best model from current experiment
BEST_MODEL_PATH = EXPERIMENT_DIR / "weights" / "best.pt"

# Check model exists
assert BEST_MODEL_PATH.exists(), (
    f"Best model not found:\n{BEST_MODEL_PATH}"
)

# Load model
model = YOLO(str(BEST_MODEL_PATH))

print("Best model loaded.")
print("Model path :", BEST_MODEL_PATH)
print("Dataset    :", DATASET_VERSION)
print("Stage      :", STAGE)

Best model loaded.
Model path : D:\PREP_INTERN\nutrivision_pro\runs\v2\segmentation\stage1\weights\best.pt
Dataset    : v2
Stage      : stage1


In [19]:
results = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    verbose=True
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-seg summary (fused): 113 layers, 2,837,298 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 272.3110.3 MB/s, size: 76.2 KB)
val: Scanning D:\PREP_INTERN\nutrivision_pro\datasets\v2\valid\labels.cache... 51 images, 0 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 51/51 8.6Mit/s 0.0s
val: D:\PREP_INTERN\nutrivision_pro\datasets\v2\valid\images\makanan_222_jpg.rf.e97d28e0a26b2bf386fa8d0a46d10625.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_INTERN\nutrivision_pro\datasets\v2\valid\images\makanan_242_jpg.rf.5db86ae95fc8fb7abddb456572d830e8.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_INTERN\nutrivision_pro\datasets\v2\valid\images\makanan_64_jpg.rf.7f7726b64ddd480e4d5f20266a72b77e.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_IN

# 8. PER CLASS PEFORMANCE

In [20]:
import pandas as pd
import numpy as np


class_names = model.names
class_indices = results.seg.ap_class_index

class_metrics = pd.DataFrame({
    "class_id": class_indices,
    "class_name": [
        class_names[int(i)]
        for i in class_indices
    ],

    "precision": results.seg.p,
    "recall": results.seg.r,
    "mAP50": results.seg.ap50,
    "mAP50-95": results.seg.ap,
})


class_metrics["f1"] = (
    2
    * class_metrics["precision"]
    * class_metrics["recall"]
    / (
        class_metrics["precision"]
        + class_metrics["recall"]
    ).replace(0, np.nan)
)


class_metrics = class_metrics.round(4)

class_metrics

,class_id,class_name,precision,recall,mAP50,mAP50-95,f1
0,0,beef,0.1238,0.3043,0.1190,0.1018,0.1761
1,1,chicken,0.1996,0.2105,0.1355,0.0771,0.2049
2,2,egg,0.1744,0.8462,0.4938,0.4345,0.2892
3,4,fruit,0.4893,0.8235,0.8034,0.5807,0.6139
4,5,noodles,0.3382,0.5000,0.4287,0.1834,0.4035
5,6,other_carbs,0.0000,0.0000,0.0075,0.0068,NaN
6,7,pork,0.0882,0.3913,0.0658,0.0349,0.1440
7,8,rice,0.6378,0.7254,0.7040,0.5080,0.6788
8,9,sambal,1.0000,0.7744,0.9950,0.9202,0.8728
9,10,seafood,0.2455,0.1176,0.0589,0.0296,0.1591


In [22]:
CSV_PATH = (
    PROJECT_ROOT
    / "runs"
    / "v2"
    / "per_class_segmentation_metrics.csv"
)

CSV_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

class_metrics.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved to:")
print(CSV_PATH.resolve())

CSV saved to:
D:\PREP_INTERN\nutrivision_pro\runs\v2\per_class_segmentation_metrics.csv
